# 01 · Spark models via Spark Connect

Run the **Spark UDF fan-out** — `groupBy(bucket).applyInPandas`, one task per `(series, model)` cell — on a real **Dataproc cluster**, driven from this notebook over a **Spark Connect** endpoint. The *same* engine code runs here as in a production Dataproc batch (the injectable-session seam, G1): the notebook just hands the engine a caller-owned session instead of letting it self-create one.

> **Reachability.** Spark Connect needs outbound access to the Dataproc endpoint. The first cell after the session does a scratch `spark.range(5).count()`; if it can't reach the endpoint, use the **remote-batch fallback** at the bottom — identical engine, submitted as a Dataproc batch.

## Get the code (cloud runtimes only)

On a cloud notebook (Colab Enterprise, Vertex Workbench) this clones or updates the repo so you're on the latest `src/`. **Skip it in a local clone** — it's a no-op guarded on the package already being importable.

In [ ]:
# Cloud bootstrap: clone + editable-install so `import scale_forecasting` resolves.
# Harmless locally — if the package already imports, we do nothing.
import importlib.util
import os
import subprocess
import sys

REPO_URL = os.environ.get("SF_REPO_URL", "https://github.com/statmike/scale-forecasting.git")
REPO_DIR = os.environ.get("SF_REPO_DIR", "scale-forecasting")

if importlib.util.find_spec("scale_forecasting") is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR], check=True)
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

## Resolve the deployment (live GCP)

`Settings.resolve()` reads the `SF_*` environment — the *same* identity every writer uses (G1), so a notebook run and a Composer run land in the same registry. Required: `SF_PROJECT_ID`, `SF_CONNECTION`, `SF_WAREHOUSE_URI`; `SF_DATASET_ID`/`SF_REGION` default. This demo targets the live `run_registry` + `v_model_leaderboard` / `v_run_summary`.

In [ ]:
from google.cloud import bigquery

from scale_forecasting.settings import Settings

settings = Settings.resolve()
client = bigquery.Client(project=settings.project_id)
DATASET = settings.dataset_ref
print("deployment:", DATASET, "region:", settings.region)

## Review helpers

Registry rows written through the Storage Write API are *async-visible*, so we poll the leaderboard briefly until a run's models show up. `leaderboard(run_id)` returns one row per model — `compute_engine` splits Spark / BigQuery / ensemble — and `run_summary(run_id)` is the header roll-up.

In [ ]:
import time

import pandas as pd


def _query(sql, run_id):
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("run_id", "STRING", run_id)]
        ),
    )
    return job.result().to_dataframe()


def leaderboard(run_id, expect_models=None, tries=12, pause=3.0):
    """Poll v_model_leaderboard until `expect_models` all appear (or give up), newest metrics."""
    sql = (
        f"SELECT model_type, compute_engine, n_cells, mean_wape, mean_mae "
        f"FROM `{DATASET}.v_model_leaderboard` WHERE run_id=@run_id "
        f"ORDER BY mean_wape"
    )
    df = pd.DataFrame()
    for _ in range(tries):
        df = _query(sql, run_id)
        if expect_models is None or set(df["model_type"]) >= set(expect_models):
            break
        time.sleep(pause)
    return df


def run_summary(run_id):
    sql = f"SELECT * FROM `{DATASET}.v_run_summary` WHERE run_id=@run_id"
    return _query(sql, run_id)

## Stand up a Spark Connect session

A `DataprocSparkSession` pinned to **runtime 3.0** (Connect needs ≥ 3.0; the batch default is left untouched). `dataproc-spark-connect` is the `[spark]` extra — a **client-only** dep, never imported by the engine (which runs on-cluster). Set `SF_DATAPROC_REGION` / `SF_DATAPROC_SUBNET` for your deployment.

In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session

region = os.environ.get("SF_DATAPROC_REGION", settings.region)
subnet = os.environ.get("SF_DATAPROC_SUBNET")  # e.g. projects/<p>/regions/<r>/subnetworks/<s>

session_cfg = Session()
session_cfg.runtime_config.version = "3.0"  # Spark Connect requires >= 3.0
if subnet:
    session_cfg.environment_config.execution_config.subnetwork_uri = subnet

spark = (
    DataprocSparkSession.builder.projectId(settings.project_id)
    .location(region)
    .dataprocSessionConfig(session_cfg)
    .getOrCreate()
)
print("Spark Connect session up:", spark.version)

## Reachability check

A trivial job proves the endpoint is reachable before we launch real work. If this raises, skip to the fallback.

> **Driver ↔ worker Python parity.** The explode fan-out ships Python to the workers via `applyInPandas`, and Spark Connect refuses to run mismatched Python minors. **Dataproc 3.0 workers run Python 3.12**, so this notebook's kernel must also be **3.12** — otherwise the run below fails with `PYTHON_VERSION_MISMATCH`. If your kernel is a different minor, use the **remote-batch fallback** at the bottom (it runs the identical engine on-cluster with no local driver, so parity is automatic).

In [ ]:
assert spark.range(5).count() == 5
print("endpoint reachable — Spark Connect is live")

## Run the explode engine over Connect

`spark_explode.run(cfg, spark=session)` runs the cross-join → bucket → `applyInPandas` fan-out **against the injected session**. Because we pass `spark`, the engine uses it and does **not** stop it (the caller owns its lifecycle). `manage_header=True` (the default) means this standalone run owns its own registry header.

In [ ]:
from scale_forecasting.config import load_config
from scale_forecasting.engines import spark_explode
from scale_forecasting.registry.ids import make_run_id

cfg = load_config("configs/explode_demo.json")
run_id = make_run_id(cfg)
print("run_id:", run_id, "| models:", cfg.models, "| method:", cfg.spark_method)

spark_explode.run(cfg, settings=settings, spark=spark)
print("explode run complete:", run_id)

## Review — Spark cells on the leaderboard

Every model ran as Spark cells (`compute_engine='spark'`) under one `run_id`.

In [ ]:
board = leaderboard(run_id, expect_models=cfg.models)
board

In [ ]:
run_summary(run_id)

## Fallback — same engine as a remote Dataproc batch

If the Connect endpoint isn't reachable from this environment, `main.run(cfg)` with **no** injected session submits the identical engine as a remote Dataproc Serverless batch. Same `run_id`, same leaderboard — the injectable-session seam means there's no second code path to trust.

In [ ]:
from scale_forecasting import main

# No `spark=` → remote Dataproc batch (the proven production path).
fallback_run_id = main.run(cfg)
leaderboard(fallback_run_id, expect_models=cfg.models)